In [ ]:
import pandas as pd

expr = pd.read_csv("../data/raw/HiSeqV2.gz", sep="\t", index_col=0)
print(expr.shape)
expr.iloc[:5, :5]

In [ ]:
expr = expr.T
print(expr.shape)
expr.iloc[:5, :5]

In [ ]:
clin = pd.read_csv("../data/raw/BRCA_clinicalMatrix", sep="\t", index_col=0, low_memory=False)
print(clin.shape)
print([c for c in clin.columns if "PAM50" in c.upper()])

In [ ]:
label_col = "PAM50Call_RNAseq"
clin[label_col].value_counts()

In [ ]:
sample = expr.index[0]
print(sample)
print(sample.split("-"))

In [ ]:
df = expr.join(clin[[label_col]], how="inner")
print(df.shape)

In [ ]:
print("before:", len(df))
df = df[df[label_col].notna()]
print("after:", len(df))

df = df.rename(columns={label_col: "subtype"})

In [ ]:
df["patient"] = ["-".join(s.split("-")[:3]) for s in df.index]
df["tss"] = [s.split("-")[1] for s in df.index]
df["sample_type"] = [s.split("-")[3][:2] for s in df.index]

df[["patient", "tss", "sample_type", "subtype"]].head()

In [ ]:
print(df["sample_type"].value_counts())
df = df[df["sample_type"] == "01"]
print(df.shape)

In [ ]:
print("samples:", len(df))
print("genes:", df.shape[1] - 4)
print("patients:", df["patient"].nunique())
print("duplicate patients:", len(df) - df["patient"].nunique())
print("hospitals:", df["tss"].nunique())
print()
print(df["subtype"].value_counts())


In [ ]:
meta = ["patient", "tss", "sample_type", "subtype"]
gene_cols = [c for c in df.columns if c not in meta]
vals = df[gene_cols].to_numpy()
print(vals.min(), vals.max(), vals.mean())

In [ ]:
df[gene_cols] = df[gene_cols].astype("float32")
df.to_parquet("../data/processed/tcga_brca.parquet")
print("saved")

In [ ]:
tets = pd.read_parquet("../data/processed/tcga_brca.parquet")
print(tets.shape)